# ERA5 : précipitations 1980–2025 sur la zone « Finistère »

**Objectif (tâche T1, groupe 1)** : charger les précipitations ERA5 sur la zone d'étude, les convertir en précipitations journalières (mm/jour) et tracer les premiers indicateurs.

**Plan du notebook**
1. Préparation (paquets, identifiants CDS)
2. Paramètres (zone, période)
3. Téléchargement des précipitations horaires, un fichier par année
4. Conversion en précipitations journalières
5. Contrôles et premiers indicateurs (cumul annuel, cycle saisonnier, histogramme, jours secs, extrêmes)

**Avant de commencer (une seule fois)**
- Créer un compte sur le site Copernicus CDS (cds.climate.copernicus.eu).
- Sur la page du jeu de données *ERA5 hourly data on single levels*, onglet *Download*, **accepter la licence** (sinon les requêtes sont refusées).
- Récupérer son **jeton d'accès** dans son profil (il sera demandé à l'étape 1).

⚠️ Sur Google Colab, les fichiers disparaissent à la fin de la session : monter Google Drive ou télécharger le fichier final.

## 1. Préparation

In [1]:
%pip install -q "cdsapi>=0.7.4" xarray netcdf4 pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\junio\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
# Identifiants CDS : crée le fichier ~/.cdsapirc s'il n'existe pas (le jeton n'est pas affiché).
# Vérifier le format exact sur la page « API how-to » du CDS si une erreur d'authentification apparaît.
import getpass
from pathlib import Path

rc = Path.home() / ".cdsapirc"
if rc.exists():
    print("Fichier ~/.cdsapirc déjà présent.")
else:
    token = getpass.getpass("Jeton d'accès CDS : ")
    rc.write_text(f"url: https://cds.climate.copernicus.eu/api\nkey: {token}\n")
    print("Fichier ~/.cdsapirc créé.")

Fichier ~/.cdsapirc déjà présent.


## 2. Paramètres

- La zone est donnée dans l'ordre **W, S, E, N**. Le CDS attend l'ordre **Nord, Ouest, Sud, Est** : la conversion est faite dans la cellule.
- ⚠️ Ce rectangle est **plus grand que le département du Finistère** (il couvre aussi l'Atlantique, la Manche et d'autres départements bretons) : à confirmer avec Tristan.

In [3]:
from pathlib import Path

W, S, E, N = -6.75, 46.25, -1.75, 50.25      # zone « Finistère » (W, S, E, N)
AREA = [N, W, S, E]                          # ordre CDS : Nord, Ouest, Sud, Est
YEAR_START, YEAR_END = 1980, 2025
DATASET = "reanalysis-era5-single-levels"

DATA_DIR = Path("era5_data")
RAW_DIR = DATA_DIR / "raw"                   # fichiers annuels horaires
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = DATA_DIR / f"era5_tp_daily_finistere_{YEAR_START}_{YEAR_END}.nc"

# Seuil (mm/jour) à partir duquel un jour est dit « pluvieux » : convention courante, à valider avec les tuteurs.
WET_DAY_MM = 1.0

## 3. Téléchargement (précipitations horaires)

- Une requête par année (≈ 8 760 valeurs par point de grille).
- Le CDS met les requêtes en **file d'attente** : compter de quelques dizaines de minutes à quelques heures pour 46 ans.
- Les fichiers déjà téléchargés sont ignorés : on peut **relancer la cellule** sans tout refaire.
- Un fichier supplémentaire (1 heure : le 1er janvier suivant à 00:00) permet de compléter le dernier jour, car dans ERA5 la valeur d'une heure est le cumul de l'heure *précédente*.

In [ ]:
import time

ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]
ALL_DAYS = [f"{d:02d}" for d in range(1, 32)]
ALL_HOURS = [f"{h:02d}:00" for h in range(24)]

def build_request(year, months=None, days=None, hours=None):
    return {
        "product_type": ["reanalysis"],
        "variable": ["total_precipitation"],
        "year": [str(year)],
        "month": months or ALL_MONTHS,
        "day": days or ALL_DAYS,
        "time": hours or ALL_HOURS,
        "area": AREA,
        "data_format": "netcdf",
        "download_format": "unarchived",
    }

jobs = [(RAW_DIR / f"era5_tp_{y}.nc", build_request(y)) for y in range(YEAR_START, YEAR_END + 1)]
extra_year = YEAR_END + 1
jobs.append((RAW_DIR / f"era5_tp_extra_{extra_year}-01-01.nc",
             build_request(extra_year, months=["01"], days=["01"], hours=["00:00"])))

missing = [(t, r) for t, r in jobs if not t.exists()]
print(f"{len(jobs) - len(missing)} fichiers déjà présents, {len(missing)} à télécharger.")

if missing:
    import cdsapi
    client = cdsapi.Client()
    for target, request in missing:
        part = target.with_suffix(".part")
        for attempt in range(1, 4):
            try:
                print(f"Téléchargement de {target.name} (essai {attempt}/3)...")
                client.retrieve(DATASET, request, str(part))
                part.rename(target)
                break
            except Exception as exc:
                print("  échec :", exc)
                time.sleep(30 * attempt)
        else:
            print(f"  ABANDON : {target.name} (relancer la cellule pour réessayer)")

1 fichiers déjà présents, 46 à télécharger.


2026-11-15 19:35:04,214 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


Téléchargement de era5_tp_1981.nc (essai 1/3)...


2026-11-15 19:35:04,618 INFO Request ID is 38af682c-3100-40d4-af49-8ed70b632a80
2026-11-15 19:35:04,697 INFO status has been updated to accepted
Recovering from connection error [('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))], attempt 1 of 500
Retrying in 120 seconds
2026-11-15 22:31:12,721 INFO status has been updated to successful


Téléchargement de era5_tp_1982.nc (essai 1/3)...


2026-11-15 22:31:18,884 INFO Request ID is 3ca78674-928c-43ee-81ee-1c717ab8995b
2026-11-15 22:31:18,966 INFO status has been updated to accepted
Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/retrieve/v1/jobs/3ca78674-928c-43ee-81ee-1c717ab8995b?log=True&request=True (Caused by NameResolutionError("HTTPSConnection(host='cds.climate.copernicus.eu', port=443): Failed to resolve 'cds.climate.copernicus.eu' ([Errno 11001] getaddrinfo failed)"))], attempt 1 of 500
Retrying in 120 seconds
2026-11-15 23:52:45,673 INFO status has been updated to running
2026-11-16 00:04:50,174 INFO status has been updated to successful


... (4 outputs left out)

Téléchargement de era5_tp_1985.nc (essai 1/3)...


2026-11-16 08:37:13,064 INFO Request ID is 509aca2e-9d95-4495-83d8-b54c1f0a93a1
2026-11-16 08:37:13,181 INFO status has been updated to accepted
2026-11-16 08:37:34,694 INFO status has been updated to running
2026-11-16 08:51:33,693 INFO status has been updated to successful


Téléchargement de era5_tp_1986.nc (essai 1/3)...


2026-11-16 08:51:36,420 INFO Request ID is b41984a8-14ec-44da-849a-8dcd6fe104b6
2026-11-16 08:51:36,525 INFO status has been updated to accepted


## 4. Conversion en précipitations journalières

Dans ERA5, `tp` est en **mètres** et la valeur à l'heure *h* est le **cumul de l'heure précédente** [h−1 h, h].
Le total du jour *D* est donc la somme des valeurs de 01:00 (jour *D*) à 00:00 (jour *D+1*), en UTC.
On décale le temps d'une heure, on somme par jour, on convertit en mm et on ne garde que les **jours complets** (24 valeurs).

➡️ Vérifier avec Tristan que la convention du produit qu'il fournit est la même (sinon décalage d'une heure).

In [ ]:
import pandas as pd
import xarray as xr

def open_raw(path):
    ds = xr.open_dataset(path)
    if "expver" in ds.dims:                       # données récentes (ERA5T) : fusionner les deux versions
        ds = ds.sel(expver=1).combine_first(ds.sel(expver=5))
    if "valid_time" in ds.dims:
        ds = ds.rename({"valid_time": "time"})
    return ds[["tp"]].drop_vars(["expver", "number"], errors="ignore")

files = sorted(RAW_DIR.glob("era5_tp_*.nc"))
print(len(files), "fichiers lus")
hourly = xr.concat([open_raw(f) for f in files], dim="time").sortby("time").drop_duplicates("time")

tp = hourly["tp"] * 1000.0                                    # m -> mm
tp = tp.assign_coords(time=tp["time"] - pd.Timedelta(hours=1))
counts = pd.Series(1, index=pd.DatetimeIndex(tp["time"].values)).resample("1D").sum()
complete_days = counts[counts == 24].index

daily = tp.resample(time="1D").sum().sel(time=complete_days)
daily.name = "tp_mm_jour"
daily.attrs = {"long_name": "Précipitations totales journalières (ERA5)", "units": "mm/jour",
               "note": "somme des 24 cumuls horaires de 01:00 à 00:00 UTC"}

daily.to_dataset().to_netcdf(OUT_FILE, encoding={"tp_mm_jour": {"zlib": True, "complevel": 4}})
print("Fichier écrit :", OUT_FILE)
daily

## 5. Contrôles

Quelques vérifications rapides. **Alerte** si le cumul annuel moyen est de quelques dizaines de mm ou de plusieurs dizaines de milliers de mm : erreur d'unité probable.

In [ ]:
lat = [d for d in daily.dims if d.startswith("lat")][0]
lon = [d for d in daily.dims if d.startswith("lon")][0]

print("Dimensions         :", dict(daily.sizes))
print("Période            :", str(daily.time.values[0])[:10], "->", str(daily.time.values[-1])[:10])
print("Min / max (mm/jour):", round(float(daily.min()), 2), "/", round(float(daily.max()), 2))
print("Valeurs manquantes :", int(daily.isnull().sum()))

mean_series = daily.mean(dim=[lat, lon])          # moyenne spatiale sur toute la zone
annual = mean_series.resample(time="YS").sum()
print("Cumul annuel moyen sur la zone (mm/an) :", round(float(annual.mean()), 1))

## 6. Premiers indicateurs (moyenne spatiale de la zone)

Indicateurs cités dans le compte rendu du 29/10 : cycle saisonnier, histogramme des pluies journalières, durées sans pluie, valeurs extrêmes.
Ils sont calculés ici sur la **moyenne spatiale** de la zone ; l'étape suivante sera de les calculer **par catchment** après interpolation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(annual["time"].dt.year, annual.values, marker="o")
ax[0].set_title("Cumul annuel (moyenne spatiale)")
ax[0].set_ylabel("mm/an")

monthly = mean_series.resample(time="MS").sum()
cycle = monthly.groupby("time.month").mean()
ax[1].bar(cycle["month"].values, cycle.values)
ax[1].set_title("Cycle saisonnier (moyenne mensuelle)")
ax[1].set_xlabel("mois"); ax[1].set_ylabel("mm/mois"); ax[1].set_xticks(range(1, 13))
fig.tight_layout(); plt.show()

In [ ]:
# Carte du cumul annuel moyen par point de grille
annual_map = daily.mean(dim="time") * 365.25
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(annual_map[lon], annual_map[lat], annual_map.values, shading="auto")
fig.colorbar(im, label="mm/an")
ax.set_title("Cumul annuel moyen par point de grille")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
plt.show()

In [ ]:
vals = mean_series.values
wet = vals[vals >= WET_DAY_MM]

# Histogramme des pluies journalières (jours pluvieux)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if len(wet):
    ax[0].hist(wet, bins=50)
    ax[0].set_yscale("log")
ax[0].set_title(f"Pluies journalières (jours >= {WET_DAY_MM} mm)")
ax[0].set_xlabel("mm/jour"); ax[0].set_ylabel("nombre de jours (échelle log)")

# Durées de périodes sans pluie (jours consécutifs < WET_DAY_MM)
runs, n = [], 0
for is_dry in vals < WET_DAY_MM:
    if is_dry:
        n += 1
    elif n:
        runs.append(n); n = 0
if n:
    runs.append(n)
if runs:
    ax[1].hist(runs, bins=range(1, max(runs) + 2))
    ax[1].set_yscale("log")
ax[1].set_title("Durée des périodes sans pluie")
ax[1].set_xlabel("jours consécutifs"); ax[1].set_ylabel("nombre de périodes (échelle log)")
fig.tight_layout(); plt.show()

years = len(annual)
print(f"Jours pluvieux (>= {WET_DAY_MM} mm) par an en moyenne : {len(wet) / years:.0f}")
print(f"Période sans pluie la plus longue : {max(runs, default=0)} jours")
if len(wet):
    print(f"Pluie journalière sur jours pluvieux : 90e centile = {np.percentile(wet, 90):.1f} mm, "
          f"99e centile = {np.percentile(wet, 99):.1f} mm, maximum = {wet.max():.1f} mm")

## Suite (à discuter avec le groupe et les tuteurs)
- Interpoler ces précipitations sur les catchments (taux de recouvrement pixel–catchment) avec les codes de Tristan.
- Recalculer les indicateurs **par catchment** et les comparer ensuite avec le CMIP6 passé (groupe 2).
- Valider le seuil de jour pluvieux et la liste d'indicateurs.